In [45]:
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
pd.options.display.float_format = '{:,.2f}'.format

## Xét promotion trên toàn bộ doanh thu từ trước giờ

In [46]:
query = """
    SELECT 
        oi.order_id,
        oi.product_id,
        oi.quantity,
        oi.unit_price,
        oi.discount_amount,
        p.cogs,

        -- flag có promo
        (oi.promo_id IS NOT NULL) AS has_promo,

        -- revenue
        oi.quantity * oi.unit_price AS revenue,

        -- cost
        oi.quantity * p.cogs AS cost,

        -- profit
        (oi.quantity * oi.unit_price) - (oi.quantity * p.cogs) AS profit,

        -- có bán lỗ
        (oi.unit_price < p.cogs) AS sell_loss_flag

    FROM '../../../data/transaction/order_items.csv' oi
    JOIN '../../../data/master/products.csv' p
    ON oi.product_id = p.product_id
"""

In [47]:
df = duckdb.connect().execute(query).df()
df

,order_id,product_id,quantity,unit_price,discount_amount,cogs,has_promo,revenue,cost,profit,sell_loss_flag
0,1,2400,7,"1,138.22",0.00,"1,053.80",False,"7,967.54","7,376.59",590.95,False
1,2,609,7,"10,166.25",0.00,"8,987.70",False,"71,163.75","62,913.93","8,249.82",False
2,3,396,3,"11,220.33",0.00,"10,091.01",False,"33,660.99","30,273.04","3,387.95",False
3,4,635,5,"10,639.25",0.00,"9,205.43",False,"53,196.25","46,027.15","7,169.10",False
4,6,1935,1,"1,597.84",0.00,"1,048.70",False,"1,597.84","1,048.70",549.14,False
...,...,...,...,...,...,...,...,...,...,...,...
714664,278159,2236,7,"2,235.69",0.00,"1,526.79",False,"15,649.83","10,687.51","4,962.32",False
714665,278160,2054,3,"6,874.67",0.00,"6,718.02",False,"20,624.01","20,154.06",469.95,False
714666,278161,2054,1,"7,040.98",0.00,"6,718.02",False,"7,040.98","6,718.02",322.96,False
714667,278162,1539,1,"4,573.67",0.00,"4,308.00",False,"4,573.67","4,308.00",265.67,False


In [48]:
df.groupby("has_promo")[["revenue", "cost", "profit"]].mean()

,revenue,cost,profit
has_promo,,,
False,"25,082.61","20,075.20","5,007.41"
True,"19,671.09","19,410.48",260.61


In [49]:
df.groupby("has_promo")["profit"].sum()

has_promo
False   2,195,015,385.58
True       72,010,680.83
Name: profit, dtype: float64

In [50]:
df.groupby("has_promo").size()

has_promo
False    438353
True     276316
dtype: int64

In [51]:
print("Tỷ lệ sử dụng promotion = ", 276316/(438353+276316)*100, "%")
print("Tỷ lệ doanh thu từ promotion = ", 72010681/(72010681+2195015386)*100, "%")

Tỷ lệ sử dụng promotion =  38.663493169565214 %
Tỷ lệ doanh thu từ promotion =  3.1764381560593677 %


Xét chung toàn bộ doanh thu từ trước giờ, tỷ lệ sử dụng promotion là `38.66%` và tỷ lệ doanh thu từ promotion là `3.17%`. Điều này cho thấy rằng mặc dù có một phần lớn khách hàng sử dụng promotion, nhưng doanh thu từ promotion chỉ chiếm một phần nhỏ trong tổng doanh thu.


=> Promotion có thể thu hút khách hàng nhưng không phải lúc nào cũng dẫn đến doanh thu cao. Có thể cần xem xét lại chiến lược promotion để tăng hiệu quả và doanh thu từ các chương trình khuyến mãi.

## Xét promotion trên doanh thu qua các năm

In [52]:
query_by_year = """
    SELECT 
        -- flag có promo
        (oi.promo_id IS NOT NULL) AS has_promo,

        -- revenue
        SUM(oi.quantity * oi.unit_price) AS sum_revenue,

        -- cost
        SUM(oi.quantity * p.cogs) AS sum_cost,

        -- profit
        SUM(oi.quantity * oi.unit_price) - SUM(oi.quantity * p.cogs) AS sum_profit,

        --năm
        EXTRACT(year FROM o.order_date) AS order_year

    FROM '../../../data/transaction/order_items.csv' oi
    JOIN '../../../data/master/products.csv' p
    ON oi.product_id = p.product_id
    JOIN '../../../data/transaction/orders.csv' o
    ON oi.order_id = o.order_id
    GROUP BY order_year, has_promo
    ORDER BY order_year, has_promo
"""

In [53]:
df_by_year = duckdb.connect().execute(query_by_year).df()
df_by_year 

,has_promo,sum_revenue,sum_cost,sum_profit,order_year
0,False,"741,497,748.02","587,461,923.81","154,035,824.21",2012
1,False,"995,286,167.07","791,789,325.43","203,496,841.64",2013
2,True,"661,883,250.20","674,190,777.88","-12,307,527.68",2013
3,False,"1,295,140,818.39","1,032,269,455.44","262,871,362.95",2014
4,True,"576,705,064.41","542,338,001.44","34,367,062.97",2014
5,False,"1,145,356,023.04","912,126,771.87","233,229,251.17",2015
6,True,"744,577,803.88","753,315,045.56","-8,737,241.68",2015
7,False,"1,483,054,466.14","1,193,058,619.98","289,995,846.16",2016
8,True,"621,586,211.40","587,500,772.62","34,085,438.78",2016
9,False,"1,178,094,637.71","944,827,938.03","233,266,699.68",2017


Có các năm profit bị âm được thấy từ `time_basic`, là các năm lẻ: 2013,2015,2017,2019,2021

Ở đây thấy rằng trong các năm đó bị âm là do promotion 

Cần xem lại chiến lược promotion trong các năm đó 

Tháng 5 doanh thu cao nhất nhưng không có promotion